In [ ]:
!pip install -q transformers numpy accelerate datasets pandas evaluate

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from datasets import Dataset
from evaluate import load
from scipy.stats import pearsonr
from google.colab import drive


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

print(os.listdir('/content/drive/MyDrive/ColabNotebooks'))


In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/train_data.csv')
test_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/test_data.csv')

test_df["clean_word"] = (
    test_df["word"].fillna("")
    .str.replace("ţ", "ț")
    .str.replace("ş", "ș")
    .str.replace("Ţ", "Ț")
    .str.replace("Ş", "Ș")
)

In [ ]:
MODEL_NAME = "readerbench/RoGPT2-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

model.eval()


In [ ]:
test_df["group_key"] = test_df["word_id"].astype(str).str.rsplit("_", n=1).str[0]
test_df["word_idx"] = test_df["word_id"].astype(str).str.rsplit("_", n=1).str[1].astype(int)

In [ ]:
unique_words = (
    test_df[["group_key", "word_idx", "clean_word"]]
    .drop_duplicates()
    .sort_values(by=["group_key", "word_idx"])
)

print(f"total test words: {len(unique_words)}")


In [ ]:
surprisal_map = {}

for group_key, group in unique_words.groupby("group_key"):
  words = group["clean_word"].tolist()
  indices = group["word_idx"].tolist()

  text, spans = "", []
  for w in words:
    spans.append((len(text), len(text) + len(w)))
    text += w + " "

  text = text.strip()

  enc = tokenizer(
      text,
      return_tensors = "pt",
      return_offsets_mapping = True,
      truncation = True,
      max_length = 1024
  )

  input_ids = enc["input_ids"].to(device)
  offsets = enc["offset_mapping"][0].tolist()

  with torch.no_grad():
    logits = model(input_ids = input_ids).logits[0]

    log_probs = torch.log_softmax(logits, dim = -1)

  # surprisal of token = -log P(token | previous token)

  tok_surp = [0.0] + [-log_probs[t - 1, input_ids[0, t]].item() for t in range(1, len(offsets))]

  for w_idx, (w_start, w_end) in zip(indices, spans):
    surprisal_map[(group_key, w_idx)] = sum(
        tok_surp[t] for t, (t_start, t_end) in enumerate(offsets)
        if t_end > w_start and t_start < w_end
    )






In [ ]:
keys = list(zip(test_df["group_key"], test_df["word_idx"]))
preds = [surprisal_map.get(k, 0.0) for k in keys]

In [ ]:
submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test_df["word_id"],
    "prediction": np.round(preds, 4)
})

OUTPUT_DIR = "/content/drive/MyDrive/ColabNotebooks/submission_rogpt2_zeroshot.csv"
submission.to_csv(OUTPUT_DIR, index = False)
print(submission.head(10))